# Reproducibility: Seed Registry and Determinism

Verifies bit-exact determinism across 3 independent runs. Tests seed registry, checkpoint round-trip.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## Determinism Verification

In [ ]:
from src.utils.seed_registry import SeedRegistry
import torch, numpy as np

def run_with_seed(seed):
    reg = SeedRegistry.get_instance()
    reg.set_master_seed(seed)
    x = torch.randn(4, 8)
    w = torch.randn(8, 4)
    return (x @ w).sum().item()

# Three independent runs with same seed
results = [run_with_seed(42) for _ in range(3)]
print(f"Run results: {results}")
assert all(abs(r - results[0]) < 1e-6 for r in results), "Non-deterministic!"
print("PASS: All 3 runs produce identical results")

# Different seeds produce different results
diff_seed = run_with_seed(99)
assert abs(diff_seed - results[0]) > 1e-6, "Seeds not affecting randomness!"
print(f"PASS: Different seed ({diff_seed:.6f}) != same seed ({results[0]:.6f})")


## Checkpoint Round-Trip

In [ ]:
import tempfile
from pathlib import Path

model = torch.nn.Linear(8, 4)
w_before = model.weight.data.clone()

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "ckpt.pt"
    torch.save({"model": model.state_dict(), "seed": 42}, path)
    model.weight.data.zero_()
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"])

assert torch.allclose(w_before, model.weight.data)
print("PASS: Checkpoint round-trip verified")
